# Research-readiness scientific audit

> This notebook audits scientifically consequential assumptions and boundaries. It does not replace the unit/integration test suite.

Work from top to bottom before freezing the schema and annotation protocol. Outputs are deliberately small, deterministic, and based on committed fixtures; this notebook does not run or select a new MAIN sample. Automated cells demonstrate implementation behavior, while the researcher remains responsible for judging linguistic validity and paper-claim adequacy.

Set the environment variable AUDIT_SMOKE_TEST=1 for the automated drift check. The scientific examples are the same in smoke mode; all model-dependent paths use the deterministic fixture transport. Live-model instructions are marked explicitly.

In [ ]:
import copy
import itertools
import json
import math
import os
import subprocess
import tempfile
from collections import Counter
from pathlib import Path

import numpy as np

from grammar_kt import canonical, folds, items, kc, kc_candidates, kc_selection, kt, normalisation, qmatrix, realisation, simulation, source, source_sampling
from grammar_kt.canonical_schema import CANONICAL_SCHEMA, SCHEMA_PATH, consistency_report
from grammar_kt.io import ROOT, read_json, read_jsonl, read_yaml, sha256_file, write_json, write_jsonl
from grammar_kt.item_diagnostic_reliability import analyse_repeated_diagnostics
from grammar_kt.item_validation import DIAGNOSTIC_PROMPT, deterministic_results, run_diagnostic
from grammar_kt.normalisation_reliability import analyse_repeated_normalisations
from grammar_kt.normalisation_validation import validate_mapping, validate_phase2_transition
from grammar_kt.realisation_space import enumerate_valid_realisations, make_valid_spec, source_conditions, subjects_for, validate_spec as validate_admissible_spec, wh_conditions
from grammar_kt.records import DIMENSIONS, FORBIDDEN_BASE_EVENT_FIELDS, grammar_cell

AUDIT_SMOKE_TEST = os.environ.get("AUDIT_SMOKE_TEST", "0") == "1"
audit_tmp = tempfile.TemporaryDirectory(prefix="grammar-kt-research-audit-")
audit_tmp_path = Path(audit_tmp.name)
settings = read_yaml(ROOT / "experiments" / "base.yaml")
frames = {row["predicate_frame_id"]: row for row in read_jsonl(realisation.LEXICON)}
selection_fixture = read_json(ROOT / "modules/kc_selection/fixtures/core.json")
selector_config = read_json(ROOT / settings["kc_selection"]["config"])
candidate_family = read_json(ROOT / selector_config["candidate_family"])
obligation_policy = read_json(ROOT / selector_config["obligation_policy"])
item_config = read_json(ROOT / settings["items"]["bank_config"])
item_template = (ROOT / settings["items"]["family_prompt"]).read_text(encoding="utf-8")
oracle_config = simulation.load_simulation_parameters(ROOT / settings["simulation"]["parameters"])
kt_config = read_json(ROOT / settings["kt"]["parameters"])

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True, default=str))

def captured_error(function):
    try:
        function()
    except Exception as error:
        return {"accepted": False, "error_type": type(error).__name__, "errors": [str(error)]}
    return {"accepted": True, "errors": []}

def nested_forbidden_keys(value, forbidden):
    found = []
    if isinstance(value, dict):
        for key, child in value.items():
            if key in forbidden:
                found.append(key)
            found.extend(nested_forbidden_keys(child, forbidden))
    elif isinstance(value, list):
        for child in value:
            found.extend(nested_forbidden_keys(child, forbidden))
    return sorted(set(found))

## 1. Research configuration

The first question is not “does the code run?” but “which exact hypothesis and parameterization am I auditing?” This cell prints the commit/dirty state and the committed scientific declarations, including their paths and version identifiers. Notebook edits themselves make the tree dirty; inspect the path list rather than treating dirtiness as a failure.

In [ ]:
git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
git_status = subprocess.run(["git", "status", "--short"], cwd=ROOT, text=True, capture_output=True, check=True).stdout.splitlines()
normalisation_method = settings["normalisation"]
normalisation_objects = {
    "phase1_prompt": {"path": normalisation_method["phase1_prompt"], "text": (ROOT / normalisation_method["phase1_prompt"]).read_text(encoding="utf-8")},
    "phase2_prompt": {"path": normalisation_method["phase2_prompt"], "text": (ROOT / normalisation_method["phase2_prompt"]).read_text(encoding="utf-8")},
    "wrapper": {"path": str(normalisation.WRAPPER.relative_to(ROOT)), "text": normalisation.WRAPPER.read_text(encoding="utf-8")},
    "rulebook": {"path": str(normalisation.RULEBOOK.relative_to(ROOT)), "sha256": sha256_file(normalisation.RULEBOOK), "text": normalisation.RULEBOOK.read_text(encoding="utf-8")},
    "mapping_schema": {"path": str(normalisation.OUTPUT_SCHEMA.relative_to(ROOT)), "object": read_json(normalisation.OUTPUT_SCHEMA)},
    "backend": {"path": normalisation_method["backend_config"], "object": read_yaml(ROOT / normalisation_method["backend_config"])},
}
configuration_under_audit = {
    "execution": {"audit_smoke_test": AUDIT_SMOKE_TEST, "git_commit": git_commit, "dirty": bool(git_status), "git_status": git_status},
    "canonical_schema": {"path": str(SCHEMA_PATH.relative_to(ROOT)), "version": CANONICAL_SCHEMA["schema_id"], "object": CANONICAL_SCHEMA},
    "kc_candidate_family": {"path": selector_config["candidate_family"], "version": candidate_family["candidate_family_id"], "object": candidate_family},
    "kc_obligation_policy": {"path": selector_config["obligation_policy"], "version": obligation_policy["obligation_policy_id"], "object": obligation_policy},
    "fold": {"path": settings["fold"]["manifest"], "version": read_json(ROOT / settings["fold"]["manifest"])["fold_id"], "object": read_json(ROOT / settings["fold"]["manifest"])},
    "simulation_oracle": {"path": settings["simulation"]["parameters"], "version": oracle_config["oracle_representation_id"], "object": oracle_config},
    "item_bank": {"path": settings["items"]["bank_config"], "version": item_config["bank_version"], "object": item_config},
    "normalisation": normalisation_objects,
    "kt": {"path": settings["kt"]["parameters"], "object": kt_config},
}
show(configuration_under_audit)

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Schema/config consistency and reference IDs have regression coverage.
- Is there a known limitation? Configuration display establishes identity, not scientific validity; every declaration still requires review.

## 2. Source evidence

Phase 1 may see only egp_id, supercategory, subcategory, guideword, and can_do. Examples and all other source fields are hidden. The temporary SHA demonstration below exercises the real source-selection guard without reading or sampling the 1,222-row MAIN source.

Pre-MAIN process concern: the committed MAIN sampling design is still an explicit zero-quota placeholder. That is correct for this audit task, but a substantive design must be preregistered before any MAIN selection.

In [ ]:
raw_source = read_jsonl(ROOT / "modules/source/fixtures/core.jsonl")[0]
visible_source = source.phase1_record(raw_source)
hidden_fields = sorted(set(raw_source) - set(source.PHASE1_FIELDS))
show({"raw_source_descriptor": raw_source, "phase1_visible_record": visible_source, "hidden_from_phase1": hidden_fields})

In [ ]:
source_demo = audit_tmp_path / "source-demo"
source_file = source_demo / "fixture-source.jsonl"
write_jsonl(source_file, [raw_source], sort_keys=False)
(source_demo / "sample_ids.txt").write_text(raw_source["egp_id"] + "\n", encoding="utf-8")
write_jsonl(source_demo / "metadata.jsonl", [{"egp_id": raw_source["egp_id"]}], sort_keys=False)
write_jsonl(source_demo / "units.jsonl", [{"unit_id": "SOURCE_AUDIT_1", "egp_id": raw_source["egp_id"], "duplicate_of": None}], sort_keys=False)
source_sha = sha256_file(source_file)
selected_source, selected_metadata, annotation_units = source.select_records(
    source_file,
    expected_sha256=source_sha,
    expected_record_count=1,
    sample_ids_path=source_demo / "sample_ids.txt",
    expected_descriptor_count=1,
    sample_metadata_path=source_demo / "metadata.jsonl",
    annotation_units_path=source_demo / "units.jsonl",
)
sha_rejection = captured_error(lambda: source.select_records(
    source_file,
    expected_sha256="0" * 64,
    expected_record_count=1,
    sample_ids_path=source_demo / "sample_ids.txt",
    expected_descriptor_count=1,
    sample_metadata_path=source_demo / "metadata.jsonl",
    annotation_units_path=source_demo / "units.jsonl",
))
show({"fixture_sha256": source_sha, "verified_selection": selected_source, "metadata": selected_metadata, "annotation_units": annotation_units, "wrong_sha_result": sha_rejection})

In [ ]:
frozen_manifest = {
    "declared_source_sha256": settings["source"]["sha256"],
    "expected_source_records": settings["source"]["records"],
    "frozen_selected_descriptor_count": settings["source"]["selected_descriptors"],
    "sample_ids_path": settings["source"]["sample_ids"],
    "first_five_frozen_ids": (ROOT / settings["source"]["sample_ids"]).read_text(encoding="utf-8").splitlines()[:5],
    "sample_metadata_path": settings["source"]["sample_metadata"],
    "annotation_units_path": settings["source"]["annotation_units"],
    "annotation_unit_count": len(read_jsonl(ROOT / settings["source"]["annotation_units"])),
}
audit_records = [
    {"egp_id": "B", "supercategory": "VERBS", "level": "B2"},
    {"egp_id": "A", "supercategory": "VERBS", "level": "B1"},
    {"egp_id": "C", "supercategory": "NOUNS", "level": "B1"},
]
audit_design = {
    "design_id": "AUDIT_ONLY_DETERMINISM",
    "allowed": {"supercategory": ["VERBS"]},
    "ordering": ["egp_id"],
    "strata": [{"stratum_id": "verbs", "match": {}, "minimum": 1, "quota": 1, "rationale": "small deterministic audit fixture"}],
}
sample_a = source_sampling.sample_records(copy.deepcopy(audit_records), audit_design)
sample_b = source_sampling.sample_records(list(reversed(audit_records)), audit_design)
assert sample_a == sample_b
show({"frozen_source_manifest": frozen_manifest, "committed_MAIN_design_template_not_executed": read_json(ROOT / "modules/source/sampling_designs/template_v0.json"), "audit_only_design": audit_design, "deterministic_selection": sample_a})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Source SHA, count, ordering, and deterministic sampling boundaries are tested.
- Is there a known limitation? A deterministic manifest can still encode a scientifically poor sampling design; scope, strata, and quotas need substantive review.

## 3. Canonical schema

GrammarCell is a six-dimensional modelling assumption, not a discovery that English grammar has exactly six cognitively atomic dimensions.

In [ ]:
show({
    "schema_id": CANONICAL_SCHEMA["schema_id"],
    "description": CANONICAL_SCHEMA["description"],
    "dimension_order": CANONICAL_SCHEMA["dimension_order"],
    "dimensions": CANONICAL_SCHEMA["dimensions"],
    "cross_field_constraints": CANONICAL_SCHEMA["cross_field_constraints"],
    "schema_mirror_consistency": consistency_report(),
})

In [ ]:
ordinary_cell = {"tense": "present", "aspect": "none", "voice": "active", "polarity": "positive", "clause": "declarative", "modal": "none"}
modal_cell = {"tense": "NA", "aspect": "none", "voice": "active", "polarity": "positive", "clause": "declarative", "modal": "would"}
imperative_cell = {"tense": "NA", "aspect": "none", "voice": "active", "polarity": "positive", "clause": "imperative", "modal": "none"}
invalid_modal_past = {**ordinary_cell, "tense": "past", "modal": "would"}
invalid_imperative = {**ordinary_cell, "clause": "imperative"}
cell_validation = []
for label, value in [
    ("valid ordinary finite", ordinary_cell),
    ("valid modal", modal_cell),
    ("valid imperative", imperative_cell),
    ("invalid modal+past", invalid_modal_past),
    ("invalid imperative configuration", invalid_imperative),
]:
    result = captured_error(lambda value=value: grammar_cell(copy.deepcopy(value), label=label))
    cell_validation.append({"case": label, "cell": value, **result})
show(cell_validation)

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Cross-field constraints and schema/prompt/structured-output consistency are tested.
- Is there a known limitation? Schema-valid cells can still omit linguistically important structure by design.

## 4. Normalisation

This is an audit of the actual two-phase boundary. Phase 1 receives only the five visible descriptor fields. Phase 2 receives those same fields, the frozen Phase-1 mapping, and examples, and it may refine only dimensions explicitly named as eligible.

The automated path below uses fixture_file. For a manual live audit, explicitly replace fixture_backend with read_yaml(ROOT / settings["normalisation"]["backend_config"]) and choose a fresh evidence directory. Never do that in the smoke test.

Pre-MAIN reproducibility concern: the committed live backend currently declares model_snapshot_pinned=false and decoding_parameters_pinned=false. The audit does not silently change that choice; the researcher must decide whether the retained evidence and repeat design are sufficient before freezing the protocol.

In [ ]:
normalisation_fixture = next(row for row in read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl") if row["fixture_label"] == "exact_simple_tense")
normalisation_response = audit_tmp_path / "normalisation-complete.json"
complete_mapping = {
    "egp_id": normalisation_fixture["egp_id"],
    "result": "complete",
    "cells": [ordinary_cell],
    "note": None,
}
write_json(normalisation_response, complete_mapping)
fixture_backend = {"kind": "fixture_file", "response_file": str(normalisation_response)}
normalised_complete = normalisation.normalise_one(
    normalisation_fixture,
    phase1_template=(ROOT / normalisation_method["phase1_prompt"]).read_text(encoding="utf-8"),
    phase2_template=(ROOT / normalisation_method["phase2_prompt"]).read_text(encoding="utf-8"),
    backend_config=fixture_backend,
    max_attempts=1,
    output=audit_tmp_path / "normalisation-complete-evidence",
)
rendered_prompt_path = Path(normalised_complete["evidence_directory"]) / "phase1/attempt-01/rendered_prompt.txt"
show({
    "selected_paths": {
        "phase1_prompt": normalisation_method["phase1_prompt"],
        "phase2_prompt": normalisation_method["phase2_prompt"],
        "rulebook": str(normalisation.RULEBOOK.relative_to(ROOT)),
        "mapping_schema": str(normalisation.OUTPUT_SCHEMA.relative_to(ROOT)),
        "declared_live_backend": normalisation_method["backend_config"],
    },
    "executed_fixture_backend": fixture_backend,
    "input": normalisation_fixture,
    "effective_phase1_evidence": source.phase1_record(normalisation_fixture),
    "mapping": normalised_complete["output"],
    "validation": validate_mapping(normalised_complete["output"], normalisation_fixture["egp_id"], phase=1),
    "phase2_routing_decision": normalised_complete["phase2_routing_reason"],
    "rendered_prompt_path": str(rendered_prompt_path),
    "rendered_prompt": rendered_prompt_path.read_text(encoding="utf-8"),
})

In [ ]:
partial_input = next(row for row in read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl") if row["fixture_label"] == "closed_present_past_ambiguity")
partial_mapping = {
    "egp_id": partial_input["egp_id"],
    "result": "partial",
    "cells": [{**ordinary_cell, "tense": ["present", "past"]}],
    "note": "phase2 eligible: tense; source realization condition: examples may establish closed present/past alternatives",
}
refined_mapping = {
    "egp_id": partial_input["egp_id"],
    "result": "complete",
    "cells": [{**ordinary_cell, "tense": "present"}, {**ordinary_cell, "tense": "past"}],
    "note": partial_mapping["note"],
}
out_of_scope_input = next(row for row in read_jsonl(ROOT / "modules/normalisation/fixtures/core.jsonl") if row["fixture_label"] == "out_of_scope")
out_of_scope_mapping = {"egp_id": out_of_scope_input["egp_id"], "result": "out_of_scope", "cells": [], "note": "discourse cohesion is outside the declared verbal morphosyntax scope"}
malformed_mapping = {"egp_id": "FIX_BAD", "result": "complete", "cells": [invalid_modal_past], "note": None}
illegal_phase2 = {**refined_mapping, "cells": [{**ordinary_cell, "tense": "present", "aspect": "perfect"}]}
normalisation_cases = [
    {
        "case": "partial routed to Phase 2",
        "input": partial_input,
        "effective_phase1_evidence": source.phase1_record(partial_input),
        "mapping": partial_mapping,
        "validation": validate_mapping(partial_mapping, partial_input["egp_id"], phase=1),
        "phase2_routing_decision": partial_mapping["result"] == "partial",
        "effective_phase2_evidence": {"record": source.phase1_record(partial_input), "phase1_mapping": partial_mapping, "examples": partial_input["examples"]},
        "phase2_mapping": refined_mapping,
        "phase2_validation": validate_mapping(refined_mapping, partial_input["egp_id"], phase=2),
        "transition_validation": validate_phase2_transition(partial_mapping, refined_mapping),
    },
    {
        "case": "out of scope",
        "input": out_of_scope_input,
        "effective_phase1_evidence": source.phase1_record(out_of_scope_input),
        "mapping": out_of_scope_mapping,
        "validation": validate_mapping(out_of_scope_mapping, out_of_scope_input["egp_id"], phase=1),
        "phase2_routing_decision": False,
    },
    {
        "case": "malformed or invalid mapping rejected",
        "input": {"egp_id": "FIX_BAD"},
        "effective_phase1_evidence": {"egp_id": "FIX_BAD"},
        "mapping": malformed_mapping,
        "validation": validate_mapping(malformed_mapping, "FIX_BAD", phase=1),
        "phase2_routing_decision": False,
    },
    {
        "case": "Phase-2 illegal modification rejected",
        "input": partial_input,
        "effective_phase2_evidence": {"record": source.phase1_record(partial_input), "phase1_mapping": partial_mapping, "examples": partial_input["examples"]},
        "mapping": illegal_phase2,
        "validation": validate_mapping(illegal_phase2, partial_input["egp_id"], phase=2),
        "transition_validation": validate_phase2_transition(partial_mapping, illegal_phase2),
        "phase2_routing_decision": True,
    },
]
show(normalisation_cases)

In [ ]:
questionable_but_schema_valid = {
    "egp_id": normalisation_fixture["egp_id"],
    "result": "complete",
    "cells": [{"tense": "past", "aspect": "perfect", "voice": "passive", "polarity": "negative", "clause": "polar_question", "modal": "none"}],
    "note": None,
}
reliability_units = [
    {"unit_id": "N1", "egp_id": "E1", "duplicate_of": None},
    {"unit_id": "N2", "egp_id": "E1", "duplicate_of": "N1"},
    {"unit_id": "N3", "egp_id": "E2", "duplicate_of": None},
    {"unit_id": "N4", "egp_id": "E2", "duplicate_of": "N3"},
]
present_complete = {"egp_id": "E1", "result": "complete", "cells": [ordinary_cell], "note": None}
past_complete = {"egp_id": "E1", "result": "complete", "cells": [{**ordinary_cell, "tense": "past"}], "note": None}
e2_complete = {"egp_id": "E2", "result": "complete", "cells": [ordinary_cell], "note": None}
e2_partial = {"egp_id": "E2", "result": "partial", "cells": [{**ordinary_cell, "tense": ["present", "past"]}], "note": "phase2 eligible: tense"}
reliability_by_unit = {
    "N1": {"output": present_complete, "phase2": None},
    "N2": {"output": past_complete, "phase2": None},
    "N3": {"output": e2_complete, "phase2": None},
    "N4": {"output": e2_partial, "phase2": {}},
}
normalisation_reliability, normalisation_comparisons = analyse_repeated_normalisations(reliability_units, reliability_by_unit)
show({
    "schema_valid_but_semantically_questionable_mapping": questionable_but_schema_valid,
    "deterministic_validator_errors": validate_mapping(questionable_but_schema_valid, normalisation_fixture["egp_id"], phase=1),
    "manual_inspection_required": "The validator checks shape and declared constraints, not whether descriptor evidence justifies the mapping.",
    "reliability_summary": normalisation_reliability,
    "pairwise_comparisons": normalisation_comparisons,
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Validator, transition, prompt rendering, retained evidence, and reliability outputs are tested.
- Is there a known limitation? Yes: an LLM can produce a schema-valid but scientifically questionable mapping. Retained prompts/evidence and repeated-annotation diagnostics expose, but do not solve, that problem.

## 5. Canonicalisation

Complete scalar mappings contribute exact cells. Multiple complete cells are OR branches; identical cells deduplicate while every descriptor-to-cell edge remains. Partial and out-of-scope mappings contribute no exact cell.

In [ ]:
past_cell = {**ordinary_cell, "tense": "past"}
negative_cell_value = {**ordinary_cell, "polarity": "negative"}
canonical_mappings = [
    {"egp_id": "DESC_ONE", "result": "complete", "cells": [ordinary_cell], "note": None},
    {"egp_id": "DESC_MULTI", "result": "complete", "cells": [past_cell, negative_cell_value], "note": "two independently asserted alternatives"},
    {"egp_id": "DESC_DUPLICATE", "result": "complete", "cells": [ordinary_cell], "note": None},
    {"egp_id": "DESC_PARTIAL", "result": "partial", "cells": [{**ordinary_cell, "tense": ["present", "past"]}], "note": "phase2 eligible: tense"},
    {"egp_id": "DESC_OOS", "result": "out_of_scope", "cells": [], "note": "outside scope"},
]
canonical_demo_dir = audit_tmp_path / "canonical-demo"
write_jsonl(canonical_demo_dir / "normalisation/final_mappings.jsonl", canonical_mappings, sort_keys=False)
canonical_summary = canonical.run(canonical_demo_dir, {})
canonical_demo_cells = read_jsonl(canonical_demo_dir / "canonical/canonical_cells.jsonl")
canonical_demo_edges = read_jsonl(canonical_demo_dir / "canonical/source_cell_edges.jsonl")
canonical_attrition = read_json(canonical_demo_dir / "canonical/audit.json")
descriptor_to_cells = {
    mapping["egp_id"]: sorted(row["canonical_cell_id"] for row in canonical_demo_edges if row["egp_id"] == mapping["egp_id"])
    for mapping in canonical_mappings
}
show({"descriptor_to_canonical_cells": descriptor_to_cells, "deduplicated_cells": canonical_demo_cells, "retained_source_cell_edges": canonical_demo_edges, "attrition_audit": canonical_attrition, "stage_summary": canonical_summary})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Complete-only contribution, deduplication, source edges, and attrition audit are tested.
- Is there a known limitation? Deduplication preserves exact tuple identity and source provenance, but cannot recover information omitted by the six-dimensional schema.

## 6. Admissible realisation space

One shared validity layer governs source conditions, frames, subjects, WH roles, imperative subtypes, and RealizationSpec validation. Realisation, item construction, and KC nuisance diagnostics all consume it.

In [ ]:
fixture_cells_by_id = {row["canonical_cell_id"]: row for row in selection_fixture["canonical_cells"]}
negative_fixture_cell = fixture_cells_by_id["CELL_FIX_NEGATIVE"]
passive_fixture_cell = fixture_cells_by_id["CELL_FIX_PASSIVE"]
question_fixture_cell = fixture_cells_by_id["CELL_FIX_QUESTION"]
negative_grid = enumerate_valid_realisations(negative_fixture_cell, frames, identity_namespace="audit-shared-space")
question_wh_fixture = next(row for row in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl") if row["fixture_label"] == "object_wh_lexical_do")
imperative_space_demo = {
    "canonical_cell_id": "CELL_AUDIT_IMPERATIVE_SPACE",
    "cell": imperative_cell,
    "source_descriptor_ids": ["IMP_ORDINARY", "IMP_EMPHATIC", "IMP_LETS", "IMP_LETS_NOT", "IMP_LET_PRONOUN"],
    "source_mapping_notes": {
        "IMP_ORDINARY": None,
        "IMP_EMPHATIC": "source realization condition: emphatic-DO",
        "IMP_LETS": "source realization condition: LET'S",
        "IMP_LETS_NOT": "source realization condition: LET'S NOT",
        "IMP_LET_PRONOUN": "source realization condition: LET + third-person pronoun",
    },
}
imperative_space_rows = enumerate_valid_realisations(imperative_space_demo, frames, identity_namespace="audit-imperative-space")
show({
    "source_conditions_negative": source_conditions(negative_fixture_cell),
    "available_frames": [{"predicate_frame_id": row["predicate_frame_id"], "frame_type": row["frame_type"], "passive_compatible": row["passive_compatible"]} for row in frames.values()],
    "subjects_for_active_lexical": subjects_for(negative_fixture_cell["cell"], frames["FRAME_WRITE"], None),
    "wh_conditions_non_subject": wh_conditions(question_wh_fixture["cell"], frames["FRAME_WRITE"]),
    "source_licensed_imperative_conditions": source_conditions(imperative_space_demo),
    "validated_imperative_subtypes": sorted({row["spec"]["imperative_subtype"] for row in imperative_space_rows}),
    "imperative_validation_errors": sorted({error for row in imperative_space_rows for error in validate_admissible_spec(row["spec"], imperative_cell, row["frame"], row["source_note"])}),
    "admissible_negative_realisation_count": len(negative_grid),
    "first_three_admissible_specs": [row["spec"] for row in negative_grid[:3]],
})

In [ ]:
negative_edges = [{"canonical_cell_id": negative_fixture_cell["canonical_cell_id"], "egp_id": source_id, "source_note": note} for source_id, note in negative_fixture_cell["source_mapping_notes"].items()]
consumer_specs = {
    "realisation.build_cases": realisation.build_cases([negative_fixture_cell], negative_edges, frames)[0]["spec"],
    "items.build_item_opportunities": items.build_item_opportunities([negative_fixture_cell], frames, item_config)[0]["realization_spec"],
    "kc_candidates.nuisance_opportunities": kc_candidates.nuisance_opportunities(negative_fixture_cell, frames)[0]["realization_spec"],
}
consumer_validation = {}
for consumer, spec in consumer_specs.items():
    note = negative_fixture_cell["source_mapping_notes"].get(spec["source_descriptor_id"])
    consumer_validation[consumer] = validate_admissible_spec(spec, negative_fixture_cell["cell"], frames[spec["predicate_frame_id"]], note)
invalid_passive_spec = copy.deepcopy(next(row for row in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl") if row["fixture_label"] == "passive")["spec"])
invalid_passive_spec["predicate_frame_id"] = "FRAME_WORK"
invalid_passive_spec["subject"] = {"text": "the technician", "person": 3, "number": "singular"}
invalid_object_wh_cell = {**question_wh_fixture["cell"], "voice": "passive"}
show({
    "shared_consumer_specs": consumer_specs,
    "shared_validation_errors": consumer_validation,
    "rejected_passive_with_intransitive_frame": validate_admissible_spec(invalid_passive_spec, passive_fixture_cell["cell"], frames["FRAME_WORK"], None),
    "rejected_object_WH_in_passive": validate_admissible_spec(question_wh_fixture["spec"], invalid_object_wh_cell, frames["FRAME_WRITE"], None),
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? All three consumers and WH/passive validity boundaries are regression-tested.
- Is there a known limitation? The shared grid is still a declared finite experimental space; constructions or lexical conditions outside it are not evaluated.

## 7. Realisation

Manually inspect morphology, operator source, argument order, and auxiliary chains. Custom rows below are audit fixtures built through the same admissibility constructor; the four named WH regressions are also asserted against committed fixtures.

In [ ]:
def make_realisation_audit_case(label, cell, frame_id, subject, *, wh=None, subtype=None, source_note=None):
    grammar_cell(cell, label=label)
    cell_row = {
        "canonical_cell_id": "CELL_AUDIT_" + label.upper().replace(" ", "_").replace("-", "_"),
        "cell": cell,
        "source_descriptor_ids": ["SOURCE_" + label.upper().replace(" ", "_").replace("-", "_")],
        "source_mapping_notes": {},
    }
    source_id = cell_row["source_descriptor_ids"][0]
    cell_row["source_mapping_notes"][source_id] = source_note
    source_case = {"source_descriptor_id": source_id, "source_note": source_note, "imperative_subtype": subtype}
    spec = make_valid_spec(cell_row, frames[frame_id], source_case, subject, wh, identity_parts=("research-audit", label))
    errors = realisation.validate_spec(spec, cell, frames[frame_id], source_note)
    derivation = realisation.realise(spec, cell, frames[frame_id]) if not errors else None
    return {
        "case": label,
        "GrammarCell": cell,
        "RealizationSpec": spec,
        "surface": derivation["surface"] if derivation else None,
        "tokens": derivation["tokens"] if derivation else None,
        "auxiliary_chain": derivation["auxiliary_chain"] if derivation else None,
        "agreement_site": derivation["agreement_site"] if derivation else None,
        "operations": derivation["operations"] if derivation else None,
        "validation_errors": errors,
    }

tech = {"text": "the technician", "person": 3, "number": "singular"}
you = {"text": "you", "person": 2, "number": "singular"}
who = {"text": "who", "person": 3, "number": "singular"}
realisation_cases = [
    make_realisation_audit_case("present declarative", ordinary_cell, "FRAME_WRITE", tech),
    make_realisation_audit_case("past declarative", past_cell, "FRAME_WRITE", tech),
    make_realisation_audit_case("lexical negative requiring DO", {**ordinary_cell, "polarity": "negative"}, "FRAME_WRITE", tech),
    make_realisation_audit_case("copular negative without DO", {**ordinary_cell, "polarity": "negative"}, "FRAME_COPULAR_READY", tech),
    make_realisation_audit_case("lexical polar question requiring DO", {**ordinary_cell, "clause": "polar_question"}, "FRAME_WRITE", tech),
    make_realisation_audit_case("inherent-operator polar question", {**ordinary_cell, "clause": "polar_question"}, "FRAME_COPULAR_READY", tech),
    make_realisation_audit_case("subject WH", {**ordinary_cell, "clause": "subject_wh_question"}, "FRAME_WRITE", who, wh={"phrase": "who", "role": "subject"}),
    make_realisation_audit_case("object WH", {**ordinary_cell, "clause": "non_subject_wh_question"}, "FRAME_WRITE", tech, wh={"phrase": "what", "role": "object"}),
    make_realisation_audit_case("adjunct WH lexical", {**ordinary_cell, "clause": "non_subject_wh_question"}, "FRAME_WRITE", tech, wh={"phrase": "when", "role": "adjunct"}),
    make_realisation_audit_case("adjunct WH copular", {**past_cell, "clause": "non_subject_wh_question"}, "FRAME_COPULAR_READY", tech, wh={"phrase": "when", "role": "adjunct"}),
    make_realisation_audit_case("passive", {**past_cell, "voice": "passive"}, "FRAME_REPAIR", {"text": "the machine", "person": 3, "number": "singular"}),
    make_realisation_audit_case("perfect", {**ordinary_cell, "aspect": "perfect"}, "FRAME_WRITE", tech),
    make_realisation_audit_case("progressive", {**ordinary_cell, "aspect": "progressive"}, "FRAME_WORK", tech),
    make_realisation_audit_case("perfect-progressive", {**ordinary_cell, "aspect": "perfect_progressive"}, "FRAME_WORK", tech),
    make_realisation_audit_case("modal", modal_cell, "FRAME_LIKE", tech),
    make_realisation_audit_case("ordinary imperative", imperative_cell, "FRAME_WRITE", you, subtype="ordinary"),
    make_realisation_audit_case("source-licensed emphatic-DO imperative", imperative_cell, "FRAME_WRITE", you, subtype="emphatic_do", source_note="source realization condition: emphatic-DO"),
]
show({"declared_realisation_rules_path": "modules/realisation/rules/default.md", "cases": realisation_cases})

In [ ]:
wh_expected = {
    "subject_wh_no_inversion": "Who writes the report?",
    "object_wh_lexical_do": "What does the technician write?",
    "adjunct_wh_lexical_do": "When does the technician write the report?",
    "adjunct_wh_inherent_operator": "When was the technician ready?",
}
wh_fixture_results = []
for fixture in read_jsonl(ROOT / "modules/realisation/fixtures/core.jsonl"):
    if fixture["fixture_label"] not in wh_expected:
        continue
    frame = frames[fixture["spec"]["predicate_frame_id"]]
    errors = realisation.validate_spec(fixture["spec"], fixture["cell"], frame, fixture.get("source_note"))
    derivation = realisation.realise(fixture["spec"], fixture["cell"], frame)
    assert derivation["surface"] == wh_expected[fixture["fixture_label"]]
    wh_fixture_results.append({"fixture": fixture["fixture_label"], "expected": wh_expected[fixture["fixture_label"]], "derivation": derivation, "errors": errors})
show(wh_fixture_results)

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Morphology, auxiliary chains, DO-support, WH behavior, passive compatibility, and imperatives are tested.
- Is there a known limitation? Deterministic well-formedness within the supported lexicon is not evidence that every generated sentence is pragmatically natural or that excluded constructions are unimportant.

## 8. Fixed item bank

The item path is canonical cell → admissible realization → item opportunity → prompt/target → stable ID. The audit inventory includes a synthetic exact WH cell because the frozen reference inventory has no observed WH cell; it does not add a reference scientific result.

In [ ]:
audit_item_mappings = [
    {"egp_id": "AUDIT_ITEM_PRESENT", "result": "complete", "cells": [ordinary_cell], "note": None},
    {"egp_id": "AUDIT_ITEM_NEGATIVE", "result": "complete", "cells": [{**ordinary_cell, "polarity": "negative"}], "note": None},
    {"egp_id": "AUDIT_ITEM_WH", "result": "complete", "cells": [{**ordinary_cell, "clause": "non_subject_wh_question"}], "note": None},
]
audit_item_cells, audit_item_edges = canonical.build(audit_item_mappings)
audit_item_opportunities = items.build_item_opportunities(audit_item_cells, frames, item_config)
audit_items = items.construct_items(audit_item_opportunities, frames, item_template)
by_opportunity = {row["item_opportunity_id"]: row for row in audit_item_opportunities}
def first_item(predicate):
    return next(row for row in audit_items if predicate(row, by_opportunity[row["item_opportunity_id"]]))
item_examples = {
    "baseline": first_item(lambda row, opp: "canonical_cell_baseline" in opp["coverage_reasons"]),
    "operator_source_contrast": first_item(lambda row, opp: "operator_source_contrast" in opp["coverage_reasons"]),
    "agreement_measurement": first_item(lambda row, opp: any(reason.startswith("agreement_measurement:") for reason in opp["coverage_reasons"])),
    "WH_item": first_item(lambda row, opp: row["realization_spec"]["wh"] is not None),
}
for row in item_examples.values():
    assert items.item_identity(row) == row["item_id"]
forbidden_item_fields = {"kc_ids", "all_kc_ids", "kc_id", "canonical_split", "split", "fold_id", "holdout_kind"}
show({
    "declared_item_config": item_config,
    "path": {
        "canonical_cells": audit_item_cells,
        "opportunity_count": len(audit_item_opportunities),
        "deterministic_item_count": len(audit_items),
    },
    "representative_items": item_examples,
    "stable_identity_checks": {name: items.item_identity(row) == row["item_id"] for name, row in item_examples.items()},
    "intrinsic_item_bank_fingerprint": items.item_bank_fingerprint(audit_items),
    "forbidden_KC_or_fold_fields_found": nested_forbidden_keys(audit_items, forbidden_item_fields),
})

In [ ]:
audit_cells_by_id = {row["canonical_cell_id"]: row["cell"] for row in audit_item_cells}
audit_edge_sources = {row["canonical_cell_id"]: set(row["source_descriptor_ids"]) for row in audit_item_cells}
audit_mapping_notes = {source_id: {"egp_id": source_id, "note": row["source_mapping_notes"][source_id]} for row in audit_item_cells for source_id in row["source_descriptor_ids"]}
hard_item_results = deterministic_results(
    audit_items,
    cells=audit_cells_by_id,
    edge_sources=audit_edge_sources,
    mappings=audit_mapping_notes,
    frames=frames,
    template=item_template,
)
assert all(row["status"] == "accepted" for row in hard_item_results)
show({"deterministic_validation": hard_item_results, "all_accepted": True})

In [ ]:
diagnostic_response_path = audit_tmp_path / "item-diagnostic-response.json"
diagnostic_response = {
    "structurally_plausible": True,
    "natural": True,
    "world_knowledge_required": False,
    "unsupported_construction": False,
    "answer_ambiguity_suspected": False,
    "note": "deterministic audit fixture response; not a human judgement",
}
write_json(diagnostic_response_path, diagnostic_response)
diagnostic_units = [
    {"validation_unit_id": "AUDIT_V1", "item_id": item_examples["baseline"]["item_id"], "duplicate_of": None},
    {"validation_unit_id": "AUDIT_V2", "item_id": item_examples["baseline"]["item_id"], "duplicate_of": "AUDIT_V1"},
]
diagnostic_rows = [run_diagnostic(
    unit,
    item_examples["baseline"],
    root=audit_tmp_path / "item-diagnostics",
    prompt_template=DIAGNOSTIC_PROMPT.read_text(encoding="utf-8"),
    backend_config={"kind": "fixture_file", "response_file": str(diagnostic_response_path)},
    max_attempts=1,
) for unit in diagnostic_units]
acceptance = read_json(ROOT / settings["items"]["validation"]["acceptance"])
diagnostic_reliability, diagnostic_comparisons = analyse_repeated_diagnostics(diagnostic_units, diagnostic_rows, acceptance)
show({
    "automated_item_diagnostic": diagnostic_rows,
    "repeat_reliability": diagnostic_reliability,
    "repeat_comparisons": diagnostic_comparisons,
    "interpretation": "This model check is an automated acceptance gate after deterministic validation; it is not deterministic linguistic proof or human validation.",
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Item identity, deterministic validation, ontology/fold absence, diagnostic evidence, and repeat reliability are tested.
- Is there a known limitation? Automated diagnostic agreement can be perfectly repeatable and still wrong; prompts, answers, and diagnostic flags need manual linguistic review.

## 9. Fold independence

The reference fold is loaded but never modified. Its IDs are joined back to schema-valid cells by the same canonical identity function. A separate in-memory audit fold pair then demonstrates that fold assignment changes only runtime metadata.

In [ ]:
reference_fold = folds.load_fold(ROOT / settings["fold"]["manifest"])
dimension_values = [CANONICAL_SCHEMA["dimensions"][field]["allowed_values"] for field in CANONICAL_SCHEMA["dimension_order"]]
schema_valid_complete_cells = []
for values in itertools.product(*dimension_values):
    candidate = dict(zip(CANONICAL_SCHEMA["dimension_order"], values))
    try:
        grammar_cell(candidate)
    except ValueError:
        continue
    if candidate["clause"] != "imperative":
        if candidate["modal"] == "none" and candidate["tense"] not in {"present", "past"}:
            continue
        if candidate["modal"] != "none" and candidate["tense"] != "NA":
            continue
    schema_valid_complete_cells.append(candidate)
all_canonical_rows, _ = canonical.build([{"egp_id": "SCHEMA_ENUMERATION_FOR_AUDIT", "result": "complete", "cells": schema_valid_complete_cells, "note": None}])
reference_ids = set().union(*(reference_fold[f"{split}_cell_ids"] for split in folds.SPLITS))
reference_cells = [row for row in all_canonical_rows if row["canonical_cell_id"] in reference_ids]
assert len(reference_cells) == len(reference_ids) == 24
reference_assignment = folds.assignment_for_cells(reference_cells, reference_fold)
reference_cell_by_id = {row["canonical_cell_id"]: row["cell"] for row in reference_cells}
show({
    "reference_fold": reference_fold,
    "matched_reference_cells": len(reference_cells),
    "first_eight_assignments": [{"canonical_cell_id": cell_id, "cell": reference_cell_by_id[cell_id], "split": reference_assignment[cell_id]} for cell_id in sorted(reference_assignment)[:8]],
})

In [ ]:
audit_cell_ids = [row["canonical_cell_id"] for row in audit_item_cells]
wh_audit_cell_id = next(row["canonical_cell_id"] for row in audit_item_cells if row["cell"]["clause"] == "non_subject_wh_question")
negative_audit_cell_id = next(row["canonical_cell_id"] for row in audit_item_cells if row["cell"]["polarity"] == "negative")
fold_a = {
    "fold_id": "AUDIT_FOLD_A",
    "require_exact_inventory": True,
    "development_cell_ids": sorted(set(audit_cell_ids) - {wh_audit_cell_id}),
    "compositional_holdout_cell_ids": [wh_audit_cell_id],
    "novel_feature_holdout_cell_ids": [],
}
fold_b = {
    "fold_id": "AUDIT_FOLD_B",
    "require_exact_inventory": True,
    "development_cell_ids": sorted(set(audit_cell_ids) - {negative_audit_cell_id}),
    "compositional_holdout_cell_ids": [negative_audit_cell_id],
    "novel_feature_holdout_cell_ids": [],
}
assignment_a = folds.assignment_for_cells(audit_item_cells, fold_a)
assignment_b = folds.assignment_for_cells(audit_item_cells, fold_b)
view_a = folds.annotate_items(audit_items, assignment_a)
view_b = folds.annotate_items(audit_items, assignment_b)
keyed_a = {row["item_id"]: row for row in view_a}
keyed_b = {row["item_id"]: row for row in view_b}
fold_invariants = {
    "item_ids_equal": sorted(keyed_a) == sorted(keyed_b),
    "RealizationSpecs_equal": {key: keyed_a[key]["realization_spec"] for key in keyed_a} == {key: keyed_b[key]["realization_spec"] for key in keyed_b},
    "prompts_equal": {key: keyed_a[key]["prompt"] for key in keyed_a} == {key: keyed_b[key]["prompt"] for key in keyed_b},
    "answers_equal": {key: keyed_a[key]["accepted_answers"] for key in keyed_a} == {key: keyed_b[key]["accepted_answers"] for key in keyed_b},
    "intrinsic_bank_fingerprint_equal": items.item_bank_fingerprint(view_a) == items.item_bank_fingerprint(view_b),
    "fold_annotations_differ": [row["canonical_split"] for row in view_a] != [row["canonical_split"] for row in view_b],
}
assert all(fold_invariants.values())
show({"fold_A": fold_a, "fold_B": fold_b, "invariants": fold_invariants, "example_runtime_annotations": [{"item_id": item_id, "fold_A": keyed_a[item_id]["canonical_split"], "fold_B": keyed_b[item_id]["canonical_split"]} for item_id in sorted(keyed_a)[:8]], "conclusion": "fold assignment is experimental metadata, not item content"})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Exact fold coverage, no overlap, and item/fold independence are tested.
- Is there a known limitation? A fold can be perfectly independent of item identity yet still be a poor test of compositionality; the semantic split definitions need manual review.

## 10. KC candidate hypothesis space

This is the complete declared space the Phase-A selector is allowed to consider. Absence from this file means absence from the search, regardless of whether a linguist would consider the omitted KC plausible.

In [ ]:
compiled_candidates = kc_candidates.add_interaction_candidates(kc_candidates.canonical_candidates(candidate_family), candidate_family)
candidate_summary = Counter((row["hypothesis_group"] or "ungrouped", row["origin"], row["family"], row["granularity_rank"]) for row in compiled_candidates)
inspect_kc_ids = [
    "KC_FINITE_PRESENT", "KC_ASPECT_PERFECT", "KC_ASPECT_PROGRESSIVE",
    "KC_QUESTION_GENERIC", "KC_POLAR_QUESTION", "KC_MODAL_CENTRAL",
    "KC_MODAL_WOULD", "KC_OP_DO_SUPPORT", "KC_INT_DO_QUESTION",
]
show({
    "candidate_family_id": candidate_family["candidate_family_id"],
    "declared_candidate_config": candidate_family,
    "compiled_candidate_count": len(compiled_candidates),
    "summary_by_hypothesis_origin_family_granularity": [{"hypothesis_group": key[0], "origin": key[1], "family": key[2], "granularity_rank": key[3], "count": count} for key, count in sorted(candidate_summary.items())],
    "inspected_candidates": [row for row in compiled_candidates if row["kc_id"] in inspect_kc_ids],
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Declarative candidate compilation, modality alternatives, interactions, and stable identities are tested.
- Is there a known limitation? Candidate selection can never identify a hypothesis that was excluded from the declared family.

## 11. KC nuisance discovery

Operation-sensitive hypotheses are evaluated over every valid nuisance realization of each development cell. DO-support is the clearest case: a negative lexical predicate needs DO, while a negative copular predicate has an inherent operator.

In [ ]:
selection_partition = kc_selection.partition_inputs(selection_fixture["canonical_cells"], selection_fixture.get("realisations", []), selection_fixture["cell_splits"])
selection_discovery = kc_candidates.discover_candidates(selection_partition["development_cells"], selection_partition["development_realisations"], frames, selector_config)
compiled_by_kc = {row["kc_id"]: row for row in selection_discovery["candidates"]}
negative_nuisance = kc_candidates.nuisance_opportunities(negative_fixture_cell, frames)
do_rule = compiled_by_kc["KC_OP_DO_SUPPORT"]["activation_rule"]
do_rows = []
for opportunity in negative_nuisance:
    active, evidence = kc.evaluate_rule(do_rule, opportunity)
    spec = opportunity["realization_spec"]
    do_rows.append({
        "realization_id": spec["realization_id"],
        "frame": spec["predicate_frame_id"],
        "frame_type": frames[spec["predicate_frame_id"]]["frame_type"],
        "subject": spec["subject"],
        "operation_evidence": opportunity["operation_facts"],
        "candidate_active": active,
        "rule_evidence": evidence,
    })
do_state = "always" if all(row["candidate_active"] for row in do_rows) else "never" if not any(row["candidate_active"] for row in do_rows) else "mixed"
diagnostics_by_kc = {row["kc_id"]: row for row in selection_discovery["diagnostics"]}
def discovered_state(kc_id, cell_id):
    return next(row["activation_state"] for row in selection_discovery["activations"] if row["kc_id"] == kc_id and row["canonical_cell_id"] == cell_id)
always_never_mixed = {
    "always": discovered_state("KC_NEGATION", "CELL_FIX_NEGATIVE"),
    "never": discovered_state("KC_FINITE_PAST", "CELL_FIX_NEGATIVE"),
    "mixed": discovered_state("KC_OP_DO_SUPPORT", "CELL_FIX_NEGATIVE"),
}
assert always_never_mixed == {"always": "always", "never": "never", "mixed": "mixed"}
show({
    "cell": negative_fixture_cell,
    "nuisance_realisation_count": len(do_rows),
    "first_twelve_realisations": do_rows[:12],
    "always_never_mixed_examples": always_never_mixed,
    "DO_support_activation_state": do_state,
    "DO_support_diagnostic": diagnostics_by_kc["KC_OP_DO_SUPPORT"],
    "cell_scope_comparison_BE_passive": diagnostics_by_kc["KC_OP_BE_PASSIVE"],
    "interpretation": {"mixed": "realisation-scope", "always_or_never": "cell-scope only when supported across the development grid"},
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Grid validity, DO-support mixed scope, and cell-scope operation diagnostics are tested.
- Is there a known limitation? The scope conclusion is conditional on the declared nuisance grid and lexicon; unenumerated realizations could change it.

## 12. KC equivalence and identifiability

Identical development activation vectors make candidates observationally indistinguishable for selection. The deterministic representative is a tie-break, not proof that its granularity is scientifically correct.

In [ ]:
question_equivalence = next(row for row in selection_discovery["equivalence_classes"] if {"KC_QUESTION_GENERIC", "KC_POLAR_QUESTION", "KC_OP_OPERATOR_INVERSION"} <= set(row["member_kc_ids"]))
question_candidate_ids = {row["candidate_id"] for row in selection_discovery["candidates"] if row["kc_id"] in question_equivalence["member_kc_ids"]}
question_activations = [row for row in selection_discovery["activations"] if row["candidate_id"] in question_candidate_ids]
question_diagnostics = [row for row in selection_discovery["diagnostics"] if row["candidate_id"] in question_candidate_ids]
show({
    "activation_rows": question_activations,
    "candidate_activation_vectors": {row["kc_id"]: row["activation_vector"] for row in question_diagnostics},
    "equivalence_class": question_equivalence,
    "deterministic_representative": next(row["kc_id"] for row in selection_discovery["candidates"] if row["candidate_id"] == question_equivalence["representative_candidate_id"]),
    "scientific_conclusion": "granularity remains unidentifiable from development activation columns",
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Equivalence grouping, representative tie-breaks, and post-freeze separation diagnostics are tested.
- Is there a known limitation? Equivalence is defined on the pilot development design; different development evidence may separate candidates.

## 13. KC obligation policy

MARKED_OPERATIONAL_v0 treats finite tense and overt/marked structures as represented facts, while active, positive, declarative, modal=none, and aspect=none are background. These are explicit scientific assumptions, not universal linguistic truths.

In [ ]:
obligation_examples = [
    ("active positive declarative background", ordinary_cell),
    ("passive represented", {**ordinary_cell, "voice": "passive"}),
    ("negative represented", {**ordinary_cell, "polarity": "negative"}),
    ("non-declarative represented", {**ordinary_cell, "clause": "polar_question"}),
    ("modal represented", modal_cell),
    ("imperative represented", imperative_cell),
]
show({
    "declared_obligation_policy": obligation_policy,
    "derived_facts": [{"case": label, "cell": cell, "facts_or_obligations": kc_candidates.salient_facts(cell, obligation_policy)} for label, cell in obligation_examples],
    "configured_background_values": obligation_policy["background_values"],
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Fact compilation and the reference eight-KC outcome are regression-tested.
- Is there a known limitation? Marked/background choices determine what the selector is required to preserve and can materially shape the ontology.

## 14. KC selection

The actual Phase-A selector covers development cell, fact, and Hamming-one contrast obligations with deterministic lexicographic greedy choices. Candidate closure would add required interaction parents; in the bundled fixture every interaction is screened out before selection, so no closure bundle fires. Backward pruning still runs and removes any unnecessary selected KC; this fixture retains all eight.

In [ ]:
selection = kc_selection.select_inventory(selection_discovery, selector_config)
selected_policy = kc_selection.compile_policy(selection, selection_partition["development_cell_ids"])
selected_kc_ids = [row["kc_id"] for row in selection["selected_candidates"]]
expected_eight = ["KC_ASPECT_PERFECT", "KC_ASPECT_PROGRESSIVE", "KC_BE_PASSIVE", "KC_FINITE_PAST", "KC_FINITE_PRESENT", "KC_IMPERATIVE", "KC_NEGATION", "KC_QUESTION_GENERIC"]
assert selected_kc_ids == expected_eight
candidate_by_id = {row["candidate_id"]: row for row in selection_discovery["candidates"]}
diagnostic_by_id = {row["candidate_id"]: row for row in selection["diagnostics"]}
parent_effects = []
for candidate in selection_discovery["candidates"]:
    if not candidate["requires_selected_ids"]:
        continue
    parent_effects.append({
        "interaction_kc": candidate["kc_id"],
        "required_parent_kcs": [candidate_by_id[parent]["kc_id"] for parent in candidate["requires_selected_ids"]],
        "selection_eligible": diagnostic_by_id[candidate["candidate_id"]]["selection_eligible"],
        "rejection_reasons": diagnostic_by_id[candidate["candidate_id"]]["rejection_reasons"],
    })
selected_trace = [row for row in selection["selection_trace"] if row["action"] == "selected"]
pruned_trace = [row for row in selection["selection_trace"] if row["action"] == "pruned"]
show({
    "development_cells": selection_partition["development_cell_ids"],
    "held_out_cells_excluded_before_discovery": selection_partition["compositional_holdout_cell_ids"] + selection_partition["novel_feature_holdout_cell_ids"],
    "candidate_discovery": {"all_candidates": len(selection_discovery["candidates"]), "eligible_candidates": [row["kc_id"] for row in selection["diagnostics"] if row["selection_eligible"]]},
    "obligations": {"count": len(selection["obligations"]), "by_kind": dict(Counter(row["kind"] for row in selection["obligations"])), "rows": selection["obligations"]},
    "greedy_selection_trace": selected_trace,
    "closure_and_parent_effects": parent_effects,
    "backward_pruning": {"pruned_trace": pruned_trace, "interpretation": "no selected KC was removable in this fixture"},
    "final_selected_kcs": selected_kc_ids,
    "objective": selection["objective"],
    "frozen_policy": selected_policy,
})

The greedy rank prioritizes marginal obligation coverage, bundle size, rule complexity, activation extent, interaction count, support, and then KC ID. Backward deletion guarantees an inclusion-minimal feasible cover: no retained KC can be removed while preserving all obligations. It does not prove global cardinality optimality, and no paper claim should say that it does.

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Determinism, feasibility, exact eight-KC result, parent requirements, and holdout isolation are tested.
- Is there a known limitation? Selection is optimal only under its declared greedy ordering and inclusion-minimality guarantee, not globally cardinality-optimal.

## 15. Held-out structural evaluation

Only after policy freezing may held-out cell content be inspected. Compare the compositional holdout, the novel-feature holdout, and the honest development-frozen full-cell memorization baseline.

In [ ]:
selection_evaluation = kc_selection.evaluate_after_freeze(
    selected_policy,
    cells=selection_fixture["canonical_cells"],
    realisations=selection_fixture.get("realisations", []),
    partition=selection_partition,
    discovery=selection_discovery,
    frames=frames,
    config=selector_config,
    mode="structural",
)
full_cell_dev_evaluation = next(row for row in selection_evaluation["baselines"] if row["label"] == "FULL_CELL_DEV_FROZEN")
show({
    "freeze_boundary": {
        "selected_policy_fingerprint": selection_evaluation["selected_policy_fingerprint"],
        "written_before_holdout_evaluation": selection_evaluation["selected_policy_written_before_holdout_evaluation"],
        "selection_partition": selection_evaluation["selection_data_partition"],
        "held_out_cell_ids_referenced_by_policy": selection_evaluation["held_out_cell_ids_referenced_by_selected_policy"],
    },
    "split_definition_audit": selection_evaluation["split_audit"],
    "selected_policy_development": selection_evaluation["selected_ontology"]["development"],
    "compositional_holdout": selection_evaluation["selected_ontology"]["compositional_holdout"],
    "novel_feature_holdout": selection_evaluation["selected_ontology"]["novel_feature_holdout"],
    "development_frozen_full_cell_baseline": full_cell_dev_evaluation,
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Held-out non-leakage, split definitions, compositional reuse, novel-feature non-invention, and baselines are tested.
- Is there a known limitation? Coverage and fact/signature preservation are structural diagnostics under the declared facts, not evidence of human acquisition.

## 16. KC materialisation

KC selection chooses and freezes a policy. KC application separately projects that policy onto concrete accepted item realizations. A declared interaction policy can assign different operation KCs to two items from the same canonical cell.

In [ ]:
selection_opportunities = items.build_item_opportunities(selection_fixture["canonical_cells"], frames, item_config)
selection_bank_intrinsic = items.construct_items(selection_opportunities, frames, item_template)
selection_assignment = {row["canonical_cell_id"]: row["split"] for row in selection_fixture["cell_splits"]}
selection_bank = folds.annotate_items(selection_bank_intrinsic, selection_assignment)
selection_hard_results = deterministic_results(
    selection_bank_intrinsic,
    cells={row["canonical_cell_id"]: row["cell"] for row in selection_fixture["canonical_cells"]},
    edge_sources={row["canonical_cell_id"]: set(row["source_descriptor_ids"]) for row in selection_fixture["canonical_cells"]},
    mappings={source_id: {"egp_id": source_id, "note": row["source_mapping_notes"][source_id]} for row in selection_fixture["canonical_cells"] for source_id in row["source_descriptor_ids"]},
    frames=frames,
    template=item_template,
)
assert all(row["status"] == "accepted" for row in selection_hard_results)
selected_item_projection, selected_cards = kc.project_items(selection_bank, selection_fixture["canonical_cells"], selected_policy)
ordinary_projection = next(row for row in selected_item_projection if row["canonical_cell_id"] == "CELL_FIX_PERFECT_PROGRESSIVE")
ordinary_item = next(row for row in selection_bank if row["item_id"] == ordinary_projection["item_id"])
interaction_policy = kc.load_policy(ROOT / "modules/kc/policies/factorized_plus_interactions.json")
interaction_projection, interaction_cards = kc.project_items(selection_bank, selection_fixture["canonical_cells"], interaction_policy)
negative_projections = [row for row in interaction_projection if row["canonical_cell_id"] == "CELL_FIX_NEGATIVE"]
negative_lexical = next(row for row in negative_projections if "do_support" in row["realization_operations"])
negative_copular = next(row for row in negative_projections if "do_support" not in row["realization_operations"])
negative_lexical_item = next(row for row in selection_bank if row["item_id"] == negative_lexical["item_id"])
negative_copular_item = next(row for row in selection_bank if row["item_id"] == negative_copular["item_id"])
show({
    "KC_SELECTION_frozen_policy": selected_policy,
    "KC_APPLICATION_ordinary_cell_scope_item": {"accepted_item": ordinary_item, "projection": ordinary_projection},
    "declared_realisation_sensitive_policy": interaction_policy["policy_id"],
    "same_cell_lexical_item": {"accepted_item": negative_lexical_item, "projection": negative_lexical},
    "same_cell_copular_item": {"accepted_item": negative_copular_item, "projection": negative_copular},
    "different_operation_KC_assignment": {"KC_INT_DO_NEGATION_in_lexical": "KC_INT_DO_NEGATION" in negative_lexical["kc_ids"], "KC_INT_DO_NEGATION_in_copular": "KC_INT_DO_NEGATION" in negative_copular["kc_ids"]},
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Cell-scope invariance and same-cell realisation-sensitive interaction projection are tested.
- Is there a known limitation? Materialisation faithfully applies a policy; it does not validate that the selected or predefined KC semantics are cognitively correct.

## 17. Q-matrix

Q construction makes no new ontology decision. It mechanically converts the frozen item–KC projection into columns, rows, and edge records; uncovered rows and weak/redundant support remain visible diagnostics.

In [ ]:
q_columns, q_rows, q_edges, q_audit = qmatrix.build(selection_bank, selected_cards, selected_item_projection)
show({
    "matrix_columns": q_columns,
    "first_twelve_item_rows": q_rows[:12],
    "first_twelve_edges": q_edges[:12],
    "row_sums": {item_id: sum(values) for item_id, values in q_rows},
    "KC_support": q_audit["scientific_diagnostics"]["kc_item_support"],
    "uncovered_items": q_audit["scientific_diagnostics"]["uncovered_item_ids"],
    "identical_columns": q_audit["scientific_diagnostics"]["identical_q_columns"],
    "low_support_KCs": q_audit["scientific_diagnostics"]["low_support_kcs"],
    "scope_counts": {"cell": q_audit["scientific_diagnostics"]["cell_scope_edges"], "realisation": q_audit["scientific_diagnostics"]["realisation_scope_edges"]},
    "structural_audit": q_audit,
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Projection coverage, edge integrity, uncovered rows, redundancy, density, and support diagnostics are tested.
- Is there a known limitation? A structurally valid Q-matrix can still have low support, redundant columns, or scientifically undesirable coverage.

## 18. Declarative oracle projection

STRUCTURAL_ORACLE_v0 maps fixed item evidence to synthetic data-generating dimensions. These oracle features are not claimed human KCs.

In [ ]:
oracle_audit_params = copy.deepcopy(oracle_config)
oracle_audit_params.update({
    "seed": int(settings["simulation"]["seed"]),
    "learners_per_profile": 1,
    "profiles": {"audit": {"beta_alpha": 2.0, "beta_beta": 2.0, "learning_rate": 0.05}},
})
oracle_projection, oracle_feature_ids = simulation.project_oracle_items(selection_bank, selection_fixture["canonical_cells"], oracle_audit_params)
question_item = next(row for row in selection_bank if row["canonical_cell_id"] == "CELL_FIX_QUESTION" and "do_support" in row["realization_evidence"]["operations"])
question_oracle_projection = next(row for row in oracle_projection if row["item_id"] == question_item["item_id"])
question_cell = fixture_cells_by_id[question_item["canonical_cell_id"]]["cell"]
question_frame_type = next(tag.removeprefix("frame_type:") for tag in question_item["realization_evidence"]["coverage_tags"] if tag.startswith("frame_type:"))
show({
    "declared_oracle": oracle_audit_params,
    "fixed_item": question_item,
    "oracle_inputs": {
        "canonical_cell": question_cell,
        "realisation_operations": question_item["realization_evidence"]["operations"],
        "agreement_site": question_item["realization_evidence"]["agreement_site"],
        "frame_type": question_frame_type,
    },
    "rule_by_rule_activation_evidence": question_oracle_projection["activation_evidence"],
    "resulting_oracle_feature_set": question_oracle_projection["oracle_feature_ids"],
    "claim_boundary": oracle_audit_params["claim_boundary"],
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Declarative rule semantics are regression-checked against the previous structural oracle behavior.
- Is there a known limitation? The oracle is one controlled synthetic world. Conclusions may be contingent on its chosen dimensions and response dynamics.

## 19. Response simulation

The response equation uses only pre-event oracle mastery, active oracle features, hashed item difficulty, and the declared complexity penalty. Public events contain no oracle or candidate-KC fields.

In [ ]:
oracle_by_item = {row["item_id"]: row["oracle_feature_ids"] for row in oracle_projection}
item_by_id = {row["item_id"]: row for row in selection_bank}
events_per_learner = len(selection_bank) * int(oracle_audit_params["item_passes_per_learner"])
train_end, validation_end = simulation.split_boundaries(events_per_learner, oracle_audit_params["train_fraction"], oracle_audit_params["validation_fraction"])
base_events, private_oracle_events, audit_learners, audit_learner_parameters = simulation.simulate_records(
    oracle_audit_params,
    item_by_id,
    oracle_by_item,
    oracle_feature_ids,
    train_end,
    validation_end,
    target_learner="L0001",
)
public_event = base_events[0]
private_event = private_oracle_events[0]
active_features = private_event["oracle_feature_ids"]
logits = {feature_id: simulation.logit(private_event["pre_mastery"][feature_id]) for feature_id in active_features}
mean_logit = sum(logits.values()) / len(logits)
explicit_z = mean_logit - public_event["item_difficulty"] - private_event["oracle_complexity_penalty"]
explicit_sigmoid = simulation.sigmoid(explicit_z)
explicit_probability = oracle_audit_params["probability_floor"] + oracle_audit_params["probability_span"] * explicit_sigmoid
assert math.isclose(explicit_probability, private_event["response_probability"], rel_tol=0, abs_tol=1e-8)
show({
    "learner_parameter_record": audit_learner_parameters[0],
    "event_equation": {
        "active_oracle_features": active_features,
        "pre_mastery": private_event["pre_mastery"],
        "feature_logits": logits,
        "mean_logit": mean_logit,
        "item_difficulty": public_event["item_difficulty"],
        "complexity_penalty": private_event["oracle_complexity_penalty"],
        "z": explicit_z,
        "sigmoid_z": explicit_sigmoid,
        "probability": explicit_probability,
        "random_draw": private_event["random_draw"],
        "correct": public_event["correct"],
        "post_event_mastery": private_event["post_mastery"],
    },
    "observable_base_event": public_event,
    "private_oracle_interaction": private_event,
    "private_fields_in_public_event": sorted(FORBIDDEN_BASE_EVENT_FIELDS & set(public_event)),
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Equation components, deterministic chronology, mastery updates, and observable/private separation are tested.
- Is there a known limitation? Synthetic probabilities are mechanically reproducible but depend on unvalidated learner profiles, gains, difficulty range, and link function.

## 20. Compositional simulation

For each learner, acquisition contains development items only. Every held-out probe reads the same post-development oracle snapshot and applies no update. Keyed draws make outcomes independent of probe ordering.

In [ ]:
phase_d = simulation.simulate_compositional_records(oracle_audit_params, item_by_id, oracle_by_item, oracle_feature_ids)
phase_d_reordered = simulation.simulate_compositional_records(oracle_audit_params, dict(reversed(list(item_by_id.items()))), dict(reversed(list(oracle_by_item.items()))), oracle_feature_ids)
learner_id = phase_d["learners"][0]["learner_id"]
learner_acquisition = [row for row in phase_d["acquisition_events"] if row["learner_id"] == learner_id]
learner_probes = [row for row in phase_d["oracle_probe_evidence"] if row["learner_id"] == learner_id]
frozen_oracle_state = next(row for row in phase_d["learner_frozen_oracle_state"] if row["learner_id"] == learner_id)
probe_read_checks = [{"event_id": row["event_id"], "reads_frozen_state": row["frozen_post_development_mastery"] == {feature_id: frozen_oracle_state["post_development_mastery"][feature_id] for feature_id in row["oracle_feature_ids"]}, "oracle_update_applied": row["oracle_update_applied"]} for row in learner_probes]
original_probe_boundary = {row["event_id"]: (row["response_probability"], row["random_draw"]) for row in phase_d["oracle_probe_evidence"]}
reordered_probe_boundary = {row["event_id"]: (row["response_probability"], row["random_draw"]) for row in phase_d_reordered["oracle_probe_evidence"]}
assert original_probe_boundary == reordered_probe_boundary
show({
    "development_acquisition": learner_acquisition,
    "only_development_items_during_acquisition": all(row["canonical_split"] == "development" for row in learner_acquisition),
    "post_development_frozen_oracle_state": frozen_oracle_state,
    "first_probe_A": learner_probes[0],
    "first_probe_B": learner_probes[1] if len(learner_probes) > 1 else None,
    "probe_reads_and_updates": probe_read_checks,
    "changing_input_item_order_changes_probe_probability_or_draw": original_probe_boundary != reordered_probe_boundary,
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Development-only acquisition, frozen probe state, no probe updates, and keyed order-independent draws are tested.
- Is there a known limitation? The protocol isolates transfer under a frozen synthetic state; it does not model learning from test items or prove human compositionality.

## 21. KT projection and frozen state

The current frozen-probe fixture is extended with one cold-KC probe and one zero-KC probe solely to make all support paths visible. Candidate histories are learned from acquisition only; probes do not update them.

In [ ]:
kt_fixture = read_json(ROOT / "modules/kt/fixtures/compositional_probe.json")
kt_probe_events = copy.deepcopy(kt_fixture["probe_events"]) + [
    {"event_id": "PROBE_COLD", "learner_id": "L_FIX", "item_id": "ITEM_COLD", "canonical_cell_id": "CELL_COLD", "canonical_split": "novel_feature_holdout", "sequence_index": 4, "timestamp": "2027-01-01T00:04:00+00:00", "correct": 0, "item_difficulty": 0.3, "dataset_split": "test", "evaluation_role": "probe", "probe_type": "novel_feature_holdout"},
    {"event_id": "PROBE_ZERO", "learner_id": "L_FIX", "item_id": "ITEM_ZERO", "canonical_cell_id": "CELL_ZERO", "canonical_split": "compositional_holdout", "sequence_index": 5, "timestamp": "2027-01-01T00:05:00+00:00", "correct": 0, "item_difficulty": -0.1, "dataset_split": "test", "evaluation_role": "probe", "probe_type": "compositional_holdout"},
]
kt_item_projections = copy.deepcopy(kt_fixture["item_projections"]) + [
    {"item_id": "ITEM_COLD", "canonical_cell_id": "CELL_COLD", "canonical_split": "novel_feature_holdout", "kc_ids": ["KC_COLD"]},
    {"item_id": "ITEM_ZERO", "canonical_cell_id": "CELL_ZERO", "canonical_split": "compositional_holdout", "kc_ids": []},
]
kt_acquisition, kt_probes, development_supported_kcs, frozen_counts = kt.project_compositional_interactions(kt_fixture["acquisition_events"], kt_probe_events, kt_item_projections)
frozen_statistics = kt.frozen_development_statistics(kt_acquisition)
all_kc_ids = ["KC_COLD", "KC_COMPONENT"]
alpha = float(kt_config["empirical"]["alpha"])
beta = float(kt_config["empirical"]["beta"])
cold_prior = float(kt_config["compositional"]["cold_kc_prior"])
probe_features, probe_targets, empirical_predictions, zero_kc_fallback = kt.frozen_probe_features(kt_probes, all_kc_ids, frozen_statistics, alpha=alpha, beta=beta, cold_prior=cold_prior)
bkt_settings = kt_config["bkt"]
_acq_bkt, bkt_probe_predictions, frozen_bkt = kt.compositional_bkt_predictions(
    kt_acquisition, kt_probes, all_kc_ids, development_supported_kcs,
    learn=float(bkt_settings["learn"]), guess=float(bkt_settings["guess"]), slip=float(bkt_settings["slip"]),
    alpha=alpha, beta=beta, cold_prior=cold_prior,
)
covered_mask = np.asarray([bool(row["kc_ids"]) for row in kt_probes])
bkt_with_fallback = bkt_probe_predictions.copy()
bkt_with_fallback[~covered_mask] = zero_kc_fallback[~covered_mask]
kt_probe_audit_rows = []
for index, row in enumerate(kt_probes):
    kt_probe_audit_rows.append({
        "projection": row,
        "empirical_prediction": float(empirical_predictions[index]),
        "bkt_prediction_with_zero_KC_fallback": float(bkt_with_fallback[index]),
        "logistic_feature_vector": probe_features[index].tolist(),
        "feature_order": ["prior_overall_rate", "prior_active_kc_rate", "mean_log_prior_opportunities", "item_difficulty", "kc_count", *all_kc_ids],
    })
frozen_counts_before = {learner: dict(counts) for learner, counts in frozen_counts.items()}
_kt_acquisition_reordered, kt_probes_reordered, _supported_reordered, frozen_counts_reordered = kt.project_compositional_interactions(kt_fixture["acquisition_events"], list(reversed(kt_probe_events)), kt_item_projections)
assert frozen_counts_before == {learner: dict(counts) for learner, counts in frozen_counts_reordered.items()}
show({
    "declared_KT_config": kt_config,
    "KC_opportunity_history": kt_acquisition,
    "development_supported_KCs": sorted(development_supported_kcs),
    "cold_and_uncovered_probe_rows": kt_probe_audit_rows,
    "frozen_counts": frozen_counts_before,
    "frozen_empirical_counts": frozen_statistics,
    "frozen_BKT_mastery": frozen_bkt,
    "cold_KC_prior": cold_prior,
    "zero_KC_fallback": "learner-global smoothed development prior",
    "probe_order_changed_frozen_counts": frozen_counts_before != {learner: dict(counts) for learner, counts in frozen_counts_reordered.items()},
    "probe_updates_candidate_state": False,
    "logistic_note": "The exact feature vectors are shown; fitting is intentionally omitted because this tiny training fixture has one outcome class.",
})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Projection histories, support labels, priors/fallbacks, BKT freezing, and non-updating probes are tested.
- Is there a known limitation? Cold and zero-KC fallbacks are declared evaluation choices and can influence comparative metrics, especially with sparse ontologies.

## 22. Cross-ontology fixed-data invariant

Two substantially different ontologies consume the same intrinsic bank and the same acquisition/probe outcomes. Their KC projections, Q matrices, and opportunity histories may differ; data generation may not.

In [ ]:
development_frozen_full_cell = kc_selection.development_frozen_full_cell_policy(selection_partition["development_cells"], obligation_policy)
full_cell_projection, full_cell_cards = kc.project_items(selection_bank, selection_fixture["canonical_cells"], development_frozen_full_cell)
selected_q_columns, _selected_q_rows, selected_q_edges, _selected_q_audit = qmatrix.build(selection_bank, selected_cards, selected_item_projection)
full_q_columns, _full_q_rows, full_q_edges, _full_q_audit = qmatrix.build(selection_bank, full_cell_cards, full_cell_projection)
fixed_acquisition = phase_d["acquisition_events"]
fixed_probes = phase_d["compositional_probe_events"] + phase_d["novel_feature_probe_events"]
selected_acquisition_history, selected_probe_history, _selected_supported, _selected_counts = kt.project_compositional_interactions(fixed_acquisition, fixed_probes, selected_item_projection)
full_acquisition_history, full_probe_history, _full_supported, _full_counts = kt.project_compositional_interactions(fixed_acquisition, fixed_probes, full_cell_projection)
fixed_data_invariants = {
    "item_content_equal": [items.item_bank_record(row) for row in selection_bank] == [items.item_bank_record(row) for row in copy.deepcopy(selection_bank)],
    "acquisition_identities_equal": [(row["event_id"], row["learner_id"], row["item_id"]) for row in fixed_acquisition] == [(row["event_id"], row["learner_id"], row["item_id"]) for row in copy.deepcopy(fixed_acquisition)],
    "probe_identities_equal": [(row["event_id"], row["learner_id"], row["item_id"]) for row in fixed_probes] == [(row["event_id"], row["learner_id"], row["item_id"]) for row in copy.deepcopy(fixed_probes)],
    "correctness_outcomes_equal": [row["correct"] for row in fixed_acquisition + fixed_probes] == [row["correct"] for row in copy.deepcopy(fixed_acquisition + fixed_probes)],
    "difficulties_equal": [row["item_difficulty"] for row in fixed_acquisition + fixed_probes] == [row["item_difficulty"] for row in copy.deepcopy(fixed_acquisition + fixed_probes)],
    "oracle_state_at_generation_boundary_equal": phase_d["learner_frozen_oracle_state"] == copy.deepcopy(phase_d["learner_frozen_oracle_state"]),
}
representation_differences = {
    "KC_IDs_selected": sorted(row["kc_id"] for row in selected_cards),
    "KC_IDs_full_cell_dev_frozen": sorted(row["kc_id"] for row in full_cell_cards),
    "Q_columns_differ": selected_q_columns != full_q_columns,
    "Q_edges_differ": {(row["item_id"], row["kc_id"]) for row in selected_q_edges} != {(row["item_id"], row["kc_id"]) for row in full_q_edges},
    "acquisition_histories_differ": [(row["event_id"], row["kc_ids"], row["opportunity_indices"]) for row in selected_acquisition_history] != [(row["event_id"], row["kc_ids"], row["opportunity_indices"]) for row in full_acquisition_history],
    "probe_histories_differ": [(row["event_id"], row["kc_ids"], row["opportunity_indices"]) for row in selected_probe_history] != [(row["event_id"], row["kc_ids"], row["opportunity_indices"]) for row in full_probe_history],
}
assert all(fixed_data_invariants.values())
assert all(value for key, value in representation_differences.items() if key.endswith("differ"))
show({"ontology_A": selected_policy["policy_id"], "ontology_B": development_frozen_full_cell["policy_id"], "fixed_data_invariants": fixed_data_invariants, "representation_differences": representation_differences, "first_selected_probe_histories": selected_probe_history[:3], "first_full_cell_probe_histories": full_probe_history[:3]})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Six-ontology temporal/compositional invariance, Q differences, history differences, and zero-KC retention are tested.
- Is there a known limitation? Fixed-data equality isolates representation effects in this synthetic design, but cannot by itself establish external or cognitive validity.

## 23. Metrics and statistics

Tiny fixture metrics are arithmetic checks only. The paired bootstrap resamples learners—not interactions—because events from the same learner share latent state and history and are not independent sampling units.

In [ ]:
metric_targets = np.asarray([0, 1, 0, 1, 1, 0], dtype=int)
metric_predictions = np.asarray([0.15, 0.80, 0.35, 0.70, 0.60, 0.40], dtype=float)
tiny_metrics = kt.prediction_metrics(metric_targets, metric_predictions, ece_bins=3)
bootstrap_left = [
    {"event_id": f"E_{learner}_{item}", "learner_id": learner, "item_id": item, "probe_type": "compositional_holdout", "correct": correct, "empirical": probability}
    for learner, values in {
        "L1": (("I1", 1, 0.8), ("I2", 0, 0.3)),
        "L2": (("I1", 0, 0.2), ("I2", 1, 0.7)),
        "L3": (("I1", 1, 0.7), ("I2", 1, 0.6)),
    }.items()
    for item, correct, probability in values
]
bootstrap_right = [{**row, "empirical": 0.5} for row in bootstrap_left]
bootstrap_result = kt.learner_bootstrap_log_loss_difference(bootstrap_left, bootstrap_right, technique="empirical", probe_type="compositional_holdout", repetitions=200, seed=17)
show({"tiny_prediction_set": {"targets": metric_targets.tolist(), "predictions": metric_predictions.tolist()}, "metrics": tiny_metrics, "paired_learner_bootstrap": bootstrap_result, "interpretation": "Do not infer scientific performance from this tiny deterministic fixture."})

**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Metric formulas and deterministic learner-level paired bootstrap behavior are tested.
- Is there a known limitation? AUC may be undefined in one-class subsets; ECE is bin-dependent; tiny fixture estimates are not paper results.

## 24. Manual audit checklist

Edit Manual audit status after review. Allowed working labels: NOT REVIEWED, ACCEPTED, ISSUE FOUND, BLOCKER. Do not infer acceptance from green automated tests.

| Module | Scientific assumption | Manual audit status | Potential blocker | Automated test coverage | Notes |
|---|---|---|---|---|---|
| Research configuration | Exact declarations and code state identify the audited hypothesis | NOT REVIEWED | — | Config/schema consistency | Record reviewer/date |
| Source | Visible evidence, SHA identity, and frozen sampling design are appropriate | NOT REVIEWED | MAIN sampling design is still a zero-quota placeholder | Source/sampling boundaries | MAIN sample not selected here |
| Canonical schema | Six dimensions, values, constraints, scope, and exclusions are adequate | NOT REVIEWED | — | Schema constraints/mirrors | Modelling assumption |
| Normalisation | Prompt evidence isolation, Phase-2 routing, and validation are defensible | NOT REVIEWED | Live model snapshot and decoding are declared unpinned | Validator/transition/reliability | Schema-valid can still be wrong |
| Canonicalisation | Complete-only contribution, OR branching, deduplication, and attrition are correct | NOT REVIEWED | — | Canonical/attrition tests | Review descriptor-to-cell edges |
| Admissible space | Shared frames, subjects, WH roles, and subtype restrictions cover intended conditions | NOT REVIEWED | — | Consumer boundary tests | Finite declared grid |
| Realisation | Morphology, chain order, operators, WH behavior, and imperatives are correct | NOT REVIEWED | — | Realisation regressions | Review every surface manually |
| Fixed item bank | Measurement coverage, prompts, targets, identities, and diagnostics are suitable | NOT REVIEWED | — | Item/fingerprint/reliability tests | Diagnostic is not human proof |
| Fold independence | Fold is metadata and holdout definitions are scientifically meaningful | NOT REVIEWED | — | Fold independence/exact coverage | Reference fold unchanged |
| KC candidates | Declared hypothesis space contains the intended alternatives | NOT REVIEWED | — | Candidate compilation tests | Omitted hypotheses cannot win |
| KC nuisance discovery | Grid-based scope classification is an adequate RQ5 test | NOT REVIEWED | — | Scope/nuisance tests | Conditional on grid/lexicon |
| KC equivalence | Identical vectors are reported without overclaiming granularity | NOT REVIEWED | — | Equivalence tests | Tie-break is not identification |
| KC obligations | Marked/background assumptions match paper claims | NOT REVIEWED | — | Fact/eight-KC tests | Not universal truths |
| KC selection | Greedy objective and inclusion-minimal guarantee are stated accurately | NOT REVIEWED | — | Determinism/feasibility tests | Not global optimum |
| Held-out evaluation | Policy freeze precedes holdout inspection and metrics match split definitions | NOT REVIEWED | — | Leakage/split tests | Structural, not cognitive evidence |
| KC materialisation | Frozen policy application is distinct from selection | NOT REVIEWED | — | Projection tests | Review cell vs realization scope |
| Q-matrix | Q is mechanical and diagnostics expose weak coverage/redundancy | NOT REVIEWED | — | Q integrity tests | Structural PASS is not adequacy |
| Oracle projection | Synthetic dimensions and activation rules match intended DGP | NOT REVIEWED | — | Oracle parity tests | Not human KCs |
| Response simulation | Equation, parameters, updates, and privacy boundary are defensible | NOT REVIEWED | — | Simulation audit tests | Synthetic profiles unvalidated |
| Compositional simulation | Acquisition/probe freeze and order independence isolate transfer | NOT REVIEWED | — | Phase-D freeze tests | No learning from probes |
| KT | Histories, priors, cold/zero-KC fallbacks, and frozen state are appropriate | NOT REVIEWED | — | KT boundary tests | Sparse coverage affects metrics |
| Cross-ontology invariant | Ontologies compare on identical generated data | NOT REVIEWED | — | Six-ontology invariance tests | Central internal-validity claim |
| Metrics/statistics | Metrics and learner-level bootstrap match estimands | NOT REVIEWED | — | Metric/bootstrap tests | Tiny examples not results |

Final review record:

- Reviewer:
- Date:
- Commit reviewed:
- Overall status: NOT REVIEWED
- Blocking issues:
- Required follow-up:


**Audit questions**

- What scientific assumption did I inspect?
- What output should I verify manually?
- What failure would invalidate the paper?
- Is this covered by an automated test? Automated coverage is complementary and is summarized per row.
- Is there a known limitation? This checklist is deliberately unfinished until a researcher records substantive judgments.